In [9]:
from dotenv import load_dotenv
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
load_dotenv()

True

In [10]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="conversational",
    temperature=0.3,
    max_new_tokens=512,
)

model = ChatHuggingFace(llm=llm)

In [11]:
#create a state
class LLMState(TypedDict):
    question: str
    answer: str

In [12]:
def llm_qa(state: LLMState) -> LLMState:
    #extract the question from state
    question = state['question']
    #form a prompt
    prompt = f"Answer the following question: {question}"
    #ask that question to llm
    answer = model.invoke(prompt).content
    state['answer'] = answer
    return state

In [13]:
#create our graph
graph= StateGraph(LLMState)

#add nodes
graph.add_node('llm_qa',llm_qa)

#add edges
graph.add_edge(START,'llm_qa')
graph.add_edge('llm_qa',END)

#compile
workflow=graph.compile()

In [14]:
#execute
initial_state = {'question': 'What is the capital of France?'}
final_state=workflow.invoke(initial_state) # this will return {'question': 'What is the capital of France?', 'answer': 'The capital of France is Paris.'}
print(final_state)

{'question': 'What is the capital of France?', 'answer': 'The capital of France is Paris.'}
